# Nhánh 1 (Người A): Nhận thức Thị giác & Theo dấu Đa đối tượng (YOLO26 + MOT)

**Dự án:** Hệ thống Giám sát Giao thông Thị giác Máy tính (BEV & YOLO26)  
**Phân hệ:** `src/perception/` — Người A (Perception & Multi-Object Tracking)  
**Môi trường thực thi khuyến nghị:** Google Colab (GPU T4 / L4 / A100)

## Kiến trúc & Hợp đồng Dữ liệu (Data Contract)
Hệ thống được tách làm 2 nhánh độc lập giao tiếp một chiều qua Hợp đồng Dữ liệu `contracts/schema.json`:
1. **Nhánh 1 (Người A - Notebook này):** Nhận luồng video/ảnh giao thông $\rightarrow$ Tiền xử lý & chuẩn hóa nhãn 4 lớp cố định (`0: motorbike`, `1: car`, `2: bus`, `3: truck`) $\rightarrow$ Huấn luyện mô hình **YOLO26** theo chiến lược 2 giai đoạn với đầu suy luận **One-to-Many (`nms=True`)** $\rightarrow$ Theo dấu đa đối tượng (**ByteTrack** vs **BoT-SORT**) $\rightarrow$ Trích xuất **điểm chạm đất** $u_{\text{đáy}} = (x_1 + x_2)/2, v_{\text{đáy}} = y_2$ $\rightarrow$ Xuất luồng `tracks_stream.jsonl` (và `.parquet`).
2. **Nhánh 2 (Người B):** Đọc `tracks_stream.jsonl`, chiếu điểm chạm đất qua ma trận Homography $H$ ($3 \times 3$) xuống bản đồ phẳng 2D Bird's-Eye View (BEV) để đo tốc độ, đếm lưu lượng theo làn, bắt lỗi sai làn, cảnh báo bám sát (THW) và vẽ Heatmap ùn tắc.

## Chỉ tiêu Nghiệm thu (Chương 6)
- **Tầng Detection (YOLO26):** $\text{mAP@50} \ge 0.80$ (ban ngày), $\text{Recall}_{\text{motorbike}} \ge 0.75$ (ban đêm / ngã tư đông đúc), kèm bảng **Đánh giá Phân lát (Slice Evaluation: Ngày vs. Đêm)**.
- **Tầng Tracking (MOT trên tập VnTraffic):** $\text{IDF1} \ge 0.75$, $\text{HOTA} \ge 0.60$, giảm thiểu $\text{IDSW}$, tốc độ suy luận $\ge 25\text{ FPS}$ trên GPU ($\le 40\text{ ms/frame}$).


---
## Phần 1: Cài đặt Môi trường & Định nghĩa Hợp đồng Dữ liệu (`contracts/schema.json`)

### 1.1. Cài đặt thư viện & kiểm tra phần cứng GPU
Các thư viện cốt lõi:
- `ultralytics>=8.4.63`: Hỗ trợ kiến trúc **YOLO26** cùng bộ theo dấu tích hợp (`bytetrack.yaml`, `botsort.yaml`).
- `roboflow`, `kaggle`: Tải trực tiếp các tập dữ liệu giao thông Việt Nam trên Colab.
- `lapx`, `motmetrics`, `trackeval`: Giải bài toán gán tuyến tính (Linear Assignment) và tính các chỉ số MOT chuẩn (`MOTA`, `IDF1`, `HOTA`, `IDSW`).
- `jsonschema`, `pyarrow`: Kiểm chứng luồng `tracks_stream.jsonl` theo `contracts/schema.json` và xuất định dạng cột `.parquet` hiệu năng cao.


In [ ]:
import os
import sys
import subprocess
from pathlib import Path


def install_packages(packages: list[str]) -> None:
    """Install required Python packages using pip without shell magics."""
    cmd = [sys.executable, "-m", "pip", "install", "-q", *packages]
    print("Installing:", " ".join(packages))
    subprocess.check_call(cmd)


REQUIRED_PACKAGES = [
    "ultralytics>=8.4.63",
    "roboflow",
    "kaggle",
    "kagglehub",
    "lapx",
    "motmetrics",
    "trackeval",
    "jsonschema",
    "pyarrow",
    "scikit-image",
    "opencv-python-headless",
    "seaborn",
    "pandas",
    "pyyaml",
    "scipy",
]

# Uncomment the line below when running on a fresh Google Colab runtime:
# install_packages(REQUIRED_PACKAGES)

import torch
import ultralytics

print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
print(f"Ultralytics: {ultralytics.__version__}")


### 1.2. Định nghĩa Từ điển 4 Lớp Chuẩn & Hợp đồng Dữ liệu (`contracts/schema.json`)
Hợp đồng dữ liệu đóng băng giao diện giữa Người A và Người B:
- **Từ điển 4 lớp cố định:** `0: motorbike`, `1: car`, `2: bus`, `3: truck`.
- **10 trường bắt buộc cho mỗi bản ghi trong `tracks_stream.jsonl`:**
  `camera_id`, `frame_id`, `timestamp_ms`, `fps`, `track_id`, `class_id`, `class_name`, `conf`, `bbox_xyxy`, `footpoint_uv`.


In [ ]:
import json
from typing import Any
import jsonschema

# Root working directory (Colab-friendly, falls back to local workspace if present)
WORK_DIR = Path("/content/traffic_bev_a") if Path("/content").exists() else Path.cwd() / "colab_work_a"
DATA_DIR = WORK_DIR / "datasets"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
RUNS_DIR = WORK_DIR / "runs"
OUTPUTS_DIR = WORK_DIR / "outputs"
CONTRACTS_DIR = WORK_DIR / "contracts"

for d in [WORK_DIR, DATA_DIR, RAW_DIR, PROCESSED_DIR, RUNS_DIR, OUTPUTS_DIR, CONTRACTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CANONICAL_CLASSES: dict[int, str] = {
    0: "motorbike",
    1: "car",
    2: "bus",
    3: "truck",
}
CLASS_NAME_TO_ID: dict[str, int] = {v: k for k, v in CANONICAL_CLASSES.items()}

TRACK_RECORD_SCHEMA: dict[str, Any] = {
    "$schema": "http://json-schema.org/draft-07/schema#",
    "title": "TrafficBEVTrackRecord",
    "description": "Data Contract between Person A (Perception/MOT) and Person B (Geometry/BEV Analytics)",
    "type": "object",
    "additionalProperties": False,
    "required": [
        "camera_id",
        "frame_id",
        "timestamp_ms",
        "fps",
        "track_id",
        "class_id",
        "class_name",
        "conf",
        "bbox_xyxy",
        "footpoint_uv",
    ],
    "properties": {
        "camera_id": {"type": "string", "minLength": 1},
        "frame_id": {"type": "integer", "minimum": 0},
        "timestamp_ms": {"type": "number", "minimum": 0.0},
        "fps": {"type": "number", "exclusiveMinimum": 0.0},
        "track_id": {"type": "integer", "minimum": 0},
        "class_id": {"type": "integer", "enum": [0, 1, 2, 3]},
        "class_name": {"type": "string", "enum": ["motorbike", "car", "bus", "truck"]},
        "conf": {"type": "number", "minimum": 0.0, "maximum": 1.0},
        "bbox_xyxy": {
            "type": "array",
            "items": {"type": "number"},
            "minItems": 4,
            "maxItems": 4,
        },
        "footpoint_uv": {
            "type": "array",
            "items": {"type": "number"},
            "minItems": 2,
            "maxItems": 2,
        },
    },
}

SCHEMA_PATH = CONTRACTS_DIR / "schema.json"
with open(SCHEMA_PATH, "w", encoding="utf-8") as f:
    json.dump(TRACK_RECORD_SCHEMA, f, indent=2, ensure_ascii=False)

print(f"Saved Data Contract schema to: {SCHEMA_PATH}")
print("Canonical Classes:", CANONICAL_CLASSES)


---
## Phần 2: Tải Tổ hợp 3 Nhóm Dữ liệu của Người A trên Colab

Nhằm thay thế hoàn toàn các nguồn UIT cũ không truy cập được, Người A sử dụng tổ hợp 3 nhóm dữ liệu thực chiến:
1. **Tập huấn luyện chủ lực (Kaggle):** `duongtran1909/vietnamese-vehicles-dataset`
   - 366.921 nhãn phương tiện định dạng YOLO `.txt` với 4 lớp chuẩn (`motorbike`, `car`, `bus`, `truck`).
   - Chia sẵn thành 2 thư mục `daytime` (49,8% — 182.715 nhãn) và `nighttime` (50,2% — 184.206 nhãn).
   - Đặc trưng mất cân bằng lớp thực tế: `motorbike` 64,18%, `car` 26,11%, `truck` 6,66%, `bus` 3,05%.
2. **Tập bổ trợ đa góc CCTV (Roboflow Universe, CC BY 4.0):**
   - `vhv2/allcam1to6`: 4.475 ảnh từ 6 góc camera giám sát.
   - `vehicledetection-krgw0/viet-nam-traffic-26n6t` (v3): 1.244 ảnh giao thông Việt Nam.
   - `vehicle-qmmot/cctv-vietnam`: 391 ảnh góc nhìn CCTV thuần túy (chuyên dùng cho Fine-tune Giai đoạn 2).
   - *(Nguồn dự phòng)* `label-tfnxz/vietnam-vehicle-detection-f4g7z`: 2.012 ảnh.
3. **Tập video chấm điểm Tracking có Ground-Truth MOT (Zenodo):**
   - `Vehicle_Tracking.zip` (`VnTraffic`, 39,5 MB): `https://zenodo.org/records/18195750/files/Vehicle_Tracking.zip?download=1`
   - Chứa video giao thông Hà Nội (HTV) đã gán nhãn sẵn Frame ID, Track ID và Bounding Box để tính trực tiếp **MOTA, IDF1, HOTA, IDSW**.


In [ ]:
import shutil
import zipfile
import urllib.request


def setup_kaggle_credentials(kaggle_json_path: Path | None = None) -> bool:
    """Configure Kaggle API credentials from file, Colab Secrets, or environment variables."""
    target_dir = Path.home() / ".kaggle"
    target_file = target_dir / "kaggle.json"
    if target_file.exists():
        os.chmod(target_file, 0o600)
        return True

    if kaggle_json_path and kaggle_json_path.exists():
        target_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy(kaggle_json_path, target_file)
        os.chmod(target_file, 0o600)
        return True

    # Try Google Colab Secrets if running on Colab
    try:
        from google.colab import userdata  # type: ignore

        k_user = userdata.get("KAGGLE_USERNAME")
        k_key = userdata.get("KAGGLE_KEY")
        if k_user and k_key:
            os.environ["KAGGLE_USERNAME"] = k_user
            os.environ["KAGGLE_KEY"] = k_key
            return True
    except Exception:
        pass

    return bool(os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"))


def download_kaggle_primary_dataset(dest_dir: Path) -> Path:
    """Download duongtran1909/vietnamese-vehicles-dataset using kagglehub (no key needed) or Kaggle CLI."""
    out_dir = dest_dir / "kaggle_vietnamese_vehicles"
    out_dir.mkdir(parents=True, exist_ok=True)

    # Try kagglehub first (works on Colab for public datasets without kaggle.json)
    try:
        import kagglehub

        print("[Kaggle] Downloading via kagglehub (no API key required)...")
        kh_path = Path(kagglehub.dataset_download("duongtran1909/vietnamese-vehicles-dataset"))
        print(f"[Kaggle] Downloaded to: {kh_path}")
        return kh_path
    except Exception as exc:
        print(f"[Kaggle] kagglehub fallback triggered ({exc}). Trying Kaggle CLI...")

    if not setup_kaggle_credentials():
        raise RuntimeError(
            "Missing Kaggle credentials and kagglehub failed. Upload kaggle.json to ~/.kaggle/kaggle.json "
            "or set KAGGLE_USERNAME & KAGGLE_KEY in Colab Secrets."
        )

    cmd = [
        "kaggle",
        "datasets",
        "download",
        "-d",
        "duongtran1909/vietnamese-vehicles-dataset",
        "-p",
        str(out_dir),
        "--unzip",
    ]
    print("Running:", " ".join(cmd))
    subprocess.check_call(cmd)
    return out_dir


# Example invocation on Colab:
# KAGGLE_DIR = download_kaggle_primary_dataset(RAW_DIR)
KAGGLE_DIR = RAW_DIR / "kaggle_vietnamese_vehicles"
print("Kaggle target directory:", KAGGLE_DIR)


In [ ]:
ROBOFLOW_SOURCES = {
    "allcam1to6": {
        "workspace": "vhv2",
        "project": "allcam1to6",
        "version": 4,
        "role": "stage1_multi_angle",
    },
    "viet_nam_traffic_26n6t": {
        "workspace": "vehicledetection-krgw0",
        "project": "viet-nam-traffic-26n6t",
        "version": 3,
        "role": "stage1_supplement",
    },
    "cctv_vietnam": {
        "workspace": "vehicle-qmmot",
        "project": "cctv-vietnam",
        "version": 1,
        "role": "stage2_cctv_finetune",
    },
    "backup_f4g7z": {
        "workspace": "label-tfnxz",
        "project": "vietnam-vehicle-detection-f4g7z",
        "version": 1,
        "role": "optional_backup",
    },
}


def get_roboflow_api_key() -> str | None:
    """Retrieve Roboflow API key from environment or Google Colab Secrets."""
    key = os.environ.get("ROBOFLOW_API_KEY")
    if key:
        return key
    try:
        from google.colab import userdata  # type: ignore

        return userdata.get("ROBOFLOW_API_KEY")
    except Exception:
        return None


def download_roboflow_datasets(
    dest_dir: Path,
    include_backup: bool = False,
    export_format: str = "yolov8",
) -> dict[str, Path]:
    """Download Roboflow Universe CCTV datasets in YOLO format."""
    from roboflow import Roboflow

    api_key = get_roboflow_api_key()
    if not api_key:
        print("[Roboflow] ROBOFLOW_API_KEY not found -> skipping Roboflow datasets (continuing with Kaggle + Zenodo).")
        return {}

    rf = Roboflow(api_key=api_key)
    downloaded: dict[str, Path] = {}

    for name, cfg in ROBOFLOW_SOURCES.items():
        if cfg["role"] == "optional_backup" and not include_backup:
            continue
        target_loc = dest_dir / f"rf_{name}"
        project = rf.workspace(cfg["workspace"]).project(cfg["project"])
        version = project.version(cfg["version"])
        dataset = version.download(export_format, location=str(target_loc))
        downloaded[name] = Path(dataset.location)
        print(f"[Roboflow] Downloaded {name} -> {downloaded[name]}")

    return downloaded


# Example invocation on Colab:
# RF_DIRS = download_roboflow_datasets(RAW_DIR, include_backup=False)
print("Configured Roboflow sources:", list(ROBOFLOW_SOURCES.keys()))


In [ ]:
ZENODO_VNTRAFFIC_URL = "https://zenodo.org/records/18195750/files/Vehicle_Tracking.zip?download=1"


def download_zenodo_vntraffic(dest_dir: Path, url: str = ZENODO_VNTRAFFIC_URL) -> Path:
    """Download and extract VnTraffic MOT benchmark dataset from Zenodo (39.5 MB)."""
    vntraffic_dir = dest_dir / "VnTraffic"
    vntraffic_dir.mkdir(parents=True, exist_ok=True)
    zip_path = dest_dir / "Vehicle_Tracking.zip"

    if not any(vntraffic_dir.iterdir()):
        print(f"Downloading VnTraffic from {url} ...")
        urllib.request.urlretrieve(url, zip_path)
        print(f"Extracting {zip_path} ({zip_path.stat().st_size / (1024 * 1024):.2f} MB)...")
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(vntraffic_dir)
        zip_path.unlink(missing_ok=True)
        print(f"Extracted VnTraffic to {vntraffic_dir}")
    else:
        print(f"VnTraffic already exists at {vntraffic_dir}")
    return vntraffic_dir


# Example invocation on Colab:
# VNTRAFFIC_DIR = download_zenodo_vntraffic(RAW_DIR)
VNTRAFFIC_DIR = RAW_DIR / "VnTraffic"
print("VnTraffic target directory:", VNTRAFFIC_DIR)


---
## Phần 3: Chuẩn hóa Nhãn & Chia tập Chống rò rỉ dữ liệu (Preprocessing & Anti-Data Leakage)

### 3.1. Ánh xạ Nhãn về Từ điển 4 Lớp & Lọc Hộp Nhiễu (`< 8 pixel`)
Do các bộ dữ liệu Roboflow và Kaggle sử dụng tên lớp khác nhau (tiếng Việt không dấu, tiếng Anh hoặc lỗi chính tả như `motorcylce`), ta chuẩn hóa toàn bộ về 4 lớp cố định của `contracts/schema.json`:

| `class_id` | `class_name` | Các tên nhãn nguồn được gộp |
| :---: | :---: | :--- |
| `0` | `motorbike` | `motorbike`, `xe may`, `motorcycle`, `motor`, `motorcylce`, `xe_may` |
| `1` | `car` | `car`, `o to`, `oto`, `o_to`, `van`, `sedan`, `suv` |
| `2` | `bus` | `bus`, `xe bus`, `xe_bus`, `xebus` |
| `3` | `truck` | `truck`, `xe tai`, `xe_tai`, `xetai` |
| *(Xóa)* | *(Loại bỏ)* | `xe dap`, `bicycle`, `bycycle`, `bike`, `people`, `person`, `train`, ... |

Ngoài ra, mọi bounding box phải thỏa mãn:
1. Tọa độ tâm và kích thước chuẩn hóa hợp lệ trong $[0, 1]$ (cắt biên nếu lệch nhẹ, loại bỏ nếu nằm ngoài khung hoặc diện tích bằng $0$).
2. Kích thước thực tế trên ảnh gốc $w_{\text{px}} = w_{\text{norm}} \cdot W \ge 8$ và $h_{\text{px}} = h_{\text{norm}} \cdot H \ge 8$ để loại bỏ hộp nhiễu siêu nhỏ làm hỏng gradient huấn luyện.


In [ ]:
import cv2
import yaml
from collections import Counter

LABEL_SYNONYMS: dict[str, int | None] = {
    # 0: motorbike
    "motorbike": 0,
    "xe may": 0,
    "xe_may": 0,
    "xemay": 0,
    "motorcycle": 0,
    "motor": 0,
    "motorcylce": 0,  # typo in xe-vietnamese / vehicles-xqno3
    "moto": 0,
    "scooter": 0,
    # 1: car
    "car": 1,
    "o to": 1,
    "o_to": 1,
    "oto": 1,
    "van": 1,
    "sedan": 1,
    "suv": 1,
    "taxi": 1,
    # 2: bus
    "bus": 2,
    "xe bus": 2,
    "xe_bus": 2,
    "xebus": 2,
    "coach": 2,
    # 3: truck
    "truck": 3,
    "xe tai": 3,
    "xe_tai": 3,
    "xetai": 3,
    "lorry": 3,
    "container": 3,
    # Explicitly drop non-motorized / irrelevant classes
    "xe dap": None,
    "xe_dap": None,
    "xedap": None,
    "bicycle": None,
    "bycycle": None,
    "bike": None,
    "people": None,
    "person": None,
    "pedestrian": None,
    "train": None,
    "tricycle": None,
    "cyclo": None,
}


def normalize_class_name(raw_name: str) -> int | None:
    """Map a raw source class name to canonical class_id (0..3) or None to drop."""
    key = raw_name.strip().lower().replace("-", " ")
    key = " ".join(key.split())
    return LABEL_SYNONYMS.get(key, None)


def remap_and_clean_labels(
    label_path: Path,
    image_path: Path,
    source_names: list[str] | dict[int, str],
    min_box_px: float = 8.0,
) -> tuple[list[str], dict[str, int]]:
    """
    Remap YOLO .txt labels to the 4 canonical classes, validate normalized coordinates [0, 1],
    and filter out noisy boxes smaller than `min_box_px` pixels on the original image.
    """
    stats: dict[str, int] = {
        "total_raw": 0,
        "kept": 0,
        "dropped_class": 0,
        "dropped_small": 0,
        "dropped_invalid": 0,
    }
    if not label_path.exists() or not image_path.exists():
        return [], stats

    from PIL import Image
    with Image.open(image_path) as im:
        img_w, img_h = im.size
    if img_w <= 0 or img_h <= 0:
        return [], stats

    if isinstance(source_names, list):
        id_to_name = {idx: name for idx, name in enumerate(source_names)}
    else:
        id_to_name = {int(k): str(v) for k, v in source_names.items()}

    cleaned_lines: list[str] = []
    with open(label_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            stats["total_raw"] += 1

            try:
                raw_cls_id = int(float(parts[0]))
                xc, yc, bw, bh = map(float, parts[1:5])
            except ValueError:
                stats["dropped_invalid"] += 1
                continue

            raw_name = id_to_name.get(raw_cls_id, str(raw_cls_id))
            canonical_id = normalize_class_name(raw_name)
            if canonical_id is None:
                stats["dropped_class"] += 1
                continue

            # Clip slightly out-of-bounds boxes and verify validity in [0, 1]
            x1 = max(0.0, min(1.0, xc - bw / 2.0))
            y1 = max(0.0, min(1.0, yc - bh / 2.0))
            x2 = max(0.0, min(1.0, xc + bw / 2.0))
            y2 = max(0.0, min(1.0, yc + bh / 2.0))

            new_bw = x2 - x1
            new_bh = y2 - y1
            if new_bw <= 0.0 or new_bh <= 0.0:
                stats["dropped_invalid"] += 1
                continue

            # Filter noisy boxes < 8 pixels on original resolution
            if (new_bw * img_w) < min_box_px or (new_bh * img_h) < min_box_px:
                stats["dropped_small"] += 1
                continue

            new_xc = (x1 + x2) / 2.0
            new_yc = (y1 + y2) / 2.0
            cleaned_lines.append(f"{canonical_id} {new_xc:.6f} {new_yc:.6f} {new_bw:.6f} {new_bh:.6f}")
            stats["kept"] += 1

    return cleaned_lines, stats


def inspect_class_distribution(labels_dir: Path) -> dict[str, int]:
    """Count instances per canonical class across a YOLO labels directory."""
    counts = Counter({0: 0, 1: 0, 2: 0, 3: 0})
    for txt_file in labels_dir.glob("*.txt"):
        with open(txt_file, "r", encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    cid = int(parts[0])
                    if cid in counts:
                        counts[cid] += 1
    return {CANONICAL_CLASSES[k]: v for k, v in counts.items()}


print("Label remapping & <8px noise filter utilities ready.")


### 3.2. Chia tập Chống rò rỉ dữ liệu theo Cụm Cảnh quay (Anti-Data Leakage) & Lọc trùng lặp SSIM
Các ảnh cắt từ cùng một luồng camera CCTV có nền đường gần như giống hệt nhau. Nếu xáo trộn ngẫu nhiên (`random_split`), các khung hình liền kề sẽ rơi vào cả tập `train` và `val/test`, gây hiện tượng **rò rỉ dữ liệu (Data Leakage)** khiến điểm `mAP` tăng ảo.

**Giải pháp kỹ thuật:**
1. **Lọc ảnh trùng lặp gần kề (Near-Duplicate Filtering):** Tính chỉ số tương đồng cấu trúc **SSIM** trên ảnh thu nhỏ (grayscale $128 \times 128$) hoặc áp dụng bước nhảy khung hình (`frame_stride`); loại bỏ các khung hình có $\text{SSIM} \ge 0.96$ so với khung hình giữ lại gần nhất.
2. **Phân cụm theo cảnh quay (Scene-Cluster Grouping):** Nhóm ảnh theo định danh camera / tiền tố chuỗi video (ví dụ `daytime_cam01_...`, `nighttime_clip04_...`), sau đó chia nguyên cụm sang `train` (75%), `val` (15%), `test` (10%) tách biệt hoàn toàn cho từng miền `daytime` và `nighttime`.
3. **Sinh 4 tệp cấu hình YAML:**
   - `dataset_stage1.yaml`: Toàn bộ tập Train/Val tổng hợp cho Huấn luyện Giai đoạn 1.
   - `dataset_stage2_finetune.yaml`: Tập con tập trung ảnh `nighttime`, góc CCTV (`cctv-vietnam`) và các mẫu chứa `bus` / `truck` cho Fine-tuning Giai đoạn 2.
   - `dataset_daytime_val.yaml` & `dataset_nighttime_val.yaml`: Hai tập kiểm định phân lát Ngày vs. Đêm.


In [ ]:
import re
import numpy as np
from skimage.metrics import structural_similarity as ssim


def extract_scene_cluster_id(image_path: Path, lighting_tag: str) -> str:
    """
    Extract a deterministic scene/video cluster ID from filename patterns
    (e.g., 'cam01_frame_00012.jpg' -> 'daytime__cam01_frame') so adjacent frames stay together.
    """
    stem = image_path.stem
    # Strip trailing frame numbers or Roboflow '.rf.<hash>' suffixes
    stem_clean = re.sub(r"\.rf\.[a-f0-9]+$", "", stem)
    cluster_prefix = re.sub(r"[_-]?\d+$", "", stem_clean)
    if not cluster_prefix:
        cluster_prefix = stem_clean[:8]
    return f"{lighting_tag}__{cluster_prefix}"


def filter_near_duplicates_ssim(
    image_paths: list[Path],
    ssim_threshold: float = 0.96,
    frame_stride: int = 1,
    thumb_size: tuple[int, int] = (128, 128),
) -> list[Path]:
    """Keep representative frames by applying frame stride and SSIM near-duplicate filtering."""
    kept_paths: list[Path] = []
    last_thumb: np.ndarray | None = None

    for idx, img_path in enumerate(sorted(image_paths)):
        if frame_stride > 1 and (idx % frame_stride != 0):
            continue
        # IMREAD_REDUCED_GRAYSCALE_8 decodes JPEG at 1/8 resolution natively (~10x faster)
        img = cv2.imread(str(img_path), cv2.IMREAD_REDUCED_GRAYSCALE_8)
        if img is None:
            continue
        thumb = cv2.resize(img, (64, 64), interpolation=cv2.INTER_AREA)

        if last_thumb is not None:
            # Fast normalized similarity check before full SSIM
            mae = float(np.mean(np.abs(thumb.astype(np.float32) - last_thumb.astype(np.float32))))
            if mae < 2.5:
                score = float(ssim(last_thumb, thumb, data_range=255))
                if score >= ssim_threshold:
                    continue

        kept_paths.append(img_path)
        last_thumb = thumb
        if (idx + 1) % 5000 == 0:
            print(f"  [SSIM Filter] Processed {idx + 1}/{len(image_paths)} images (kept {len(kept_paths)})...")

    return kept_paths


def split_dataset_by_scene_clusters(
    records: list[dict[str, Any]],
    output_root: Path,
    train_ratio: float = 0.75,
    val_ratio: float = 0.15,
    seed: int = 42,
) -> dict[str, Path]:
    """
    Split preprocessed samples into train/val/test without scene leakage,
    and generate YAML configs for Stage 1, Stage 2 Fine-tuning, Daytime Val, and Nighttime Val.
    Each record in `records` expects: {'image': Path, 'labels': list[str], 'lighting': 'daytime'|'nighttime'|'cctv', 'source': str}.
    """
    rng = np.random.default_rng(seed)
    clusters: dict[str, list[dict[str, Any]]] = {}
    for rec in records:
        cid = extract_scene_cluster_id(rec["image"], rec["lighting"])
        clusters.setdefault(cid, []).append(rec)

    # Stratify clusters by lighting domain so daytime & nighttime are balanced across splits
    split_assignments: dict[str, list[dict[str, Any]]] = {"train": [], "val": [], "test": []}
    stage2_train: list[dict[str, Any]] = []
    stage2_val: list[dict[str, Any]] = []
    daytime_val: list[dict[str, Any]] = []
    nighttime_val: list[dict[str, Any]] = []

    for lighting in ["daytime", "nighttime", "cctv"]:
        domain_cids = [c for c in clusters if c.startswith(f"{lighting}__")]
        rng.shuffle(domain_cids)
        n_total = len(domain_cids)
        n_train = int(round(n_total * train_ratio))
        n_val = int(round(n_total * val_ratio))

        train_cids = set(domain_cids[:n_train])
        val_cids = set(domain_cids[n_train : n_train + n_val])

        for cid in domain_cids:
            items = clusters[cid]
            if cid in train_cids:
                split_name = "train"
            elif cid in val_cids:
                split_name = "val"
            else:
                split_name = "test"

            split_assignments[split_name].extend(items)

            # Slice evaluation sets
            if split_name == "val" and lighting == "daytime":
                daytime_val.extend(items)
            elif split_name == "val" and lighting == "nighttime":
                nighttime_val.extend(items)

            # Stage 2 Fine-tuning subset: nighttime + cctv + minority classes (bus=2, truck=3)
            for item in items:
                has_minority = any(line.startswith(("2 ", "3 ")) for line in item["labels"])
                is_target_domain = lighting in ("nighttime", "cctv") or item["source"] == "cctv_vietnam"
                if is_target_domain or has_minority:
                    if split_name == "train":
                        stage2_train.append(item)
                    elif split_name == "val":
                        stage2_val.append(item)

    def _materialize_split(items: list[dict[str, Any]], split_dir: Path) -> Path:
        img_dir = split_dir / "images"
        lbl_dir = split_dir / "labels"
        img_dir.mkdir(parents=True, exist_ok=True)
        lbl_dir.mkdir(parents=True, exist_ok=True)
        for item in items:
            src_img: Path = item["image"]
            dst_img = img_dir / src_img.name
            if not dst_img.exists():
                try:
                    os.symlink(src_img.resolve(), dst_img)
                except OSError:
                    shutil.copy2(src_img, dst_img)
            dst_lbl = lbl_dir / f"{src_img.stem}.txt"
            with open(dst_lbl, "w", encoding="utf-8") as f:
                f.write("\n".join(item["labels"]) + ("\n" if item["labels"] else ""))
        return split_dir

    s1_train_dir = _materialize_split(split_assignments["train"], output_root / "stage1" / "train")
    s1_val_dir = _materialize_split(split_assignments["val"], output_root / "stage1" / "val")
    s1_test_dir = _materialize_split(split_assignments["test"], output_root / "stage1" / "test")
    s2_train_dir = _materialize_split(stage2_train, output_root / "stage2" / "train")
    s2_val_dir = _materialize_split(stage2_val, output_root / "stage2" / "val")
    day_val_dir = _materialize_split(daytime_val, output_root / "slices" / "daytime_val")
    night_val_dir = _materialize_split(nighttime_val, output_root / "slices" / "nighttime_val")

    def _write_yaml(yaml_path: Path, train_p: Path, val_p: Path, test_p: Path | None = None) -> Path:
        cfg: dict[str, Any] = {
            "path": str(output_root.resolve()),
            "train": str((train_p / "images").resolve()),
            "val": str((val_p / "images").resolve()),
            "names": CANONICAL_CLASSES,
            "nc": 4,
        }
        if test_p is not None:
            cfg["test"] = str((test_p / "images").resolve())
        with open(yaml_path, "w", encoding="utf-8") as f:
            yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)
        return yaml_path

    yaml_paths = {
        "stage1": _write_yaml(output_root / "dataset_stage1.yaml", s1_train_dir, s1_val_dir, s1_test_dir),
        "stage2": _write_yaml(output_root / "dataset_stage2_finetune.yaml", s2_train_dir, s2_val_dir),
        "daytime_val": _write_yaml(output_root / "dataset_daytime_val.yaml", s1_train_dir, day_val_dir),
        "nighttime_val": _write_yaml(output_root / "dataset_nighttime_val.yaml", s1_train_dir, night_val_dir),
    }
    print("Generated YOLO dataset YAMLs:", {k: str(v) for k, v in yaml_paths.items()})
    return yaml_paths


print("Anti-data-leakage scene-cluster splitting & YAML generator ready.")


---
## Phần 4: Huấn luyện Mô hình YOLO26 theo Chiến lược 2 Giai đoạn

### 4.1. Tại sao bắt buộc giữ đầu suy luận One-to-Many (`nms=True`)?
Trong hệ sinh thái **YOLO26**, chế độ mặc định mới có thể hỗ trợ suy luận không cần NMS (One-to-One, `nms=False` / `end2end=True`), tuy nhiên chế độ này thường giới hạn số lượng đối tượng tối đa mỗi khung hình (ví dụ `max_det=300`). Tại các ngã tư giờ cao điểm ở TP.HCM và Hà Nội, mật độ xe máy có thể vượt ngưỡng dày đặc và bị che khuất một phần. Do đó, Người A **bắt buộc giữ đầu suy luận One-to-Many có lọc NMS (`nms=True`, `end2end=False`, `max_det=500`)** để tránh sót xe máy.

### 4.2. Chiến lược Huấn luyện 2 Giai đoạn & Tăng cường Dữ liệu HSV
- **Tăng cường dữ liệu nhẹ về màu sắc:** `hsv_h=0.015`, `hsv_s=0.50`, `hsv_v=0.40` giúp mô hình thích nghi với hiện tượng lóa đèn pha ban đêm và bóng râm gầm cầu vượt; tắt lật dọc (`flipud=0.0`) vì phương tiện không bao giờ lật ngược.
- **Giai đoạn 1 (Generalist Training):** Huấn luyện `yolo26s.pt` (tự động chuyển sang `yolo26n.pt` nếu chạy trên CPU) trên toàn bộ dữ liệu tổng hợp (`dataset_stage1.yaml`).
- **Giai đoạn 2 (Domain & Class-Imbalance Fine-tuning):** Nạp trọng số `best.pt` từ Giai đoạn 1, tinh chỉnh với learning rate nhỏ (`lr0=0.001`) trên tập con gồm ảnh ban đêm (`nighttime`), góc máy CCTV (`cctv-vietnam`) và các mẫu lớp thiểu số `bus` (3,05%) / `truck` (6,66%).


In [ ]:
from ultralytics import YOLO


def select_yolo26_backbone(prefer_small: bool = True) -> str:
    """Select yolo26s.pt on GPU and fall back to yolo26n.pt on CPU."""
    if torch.cuda.is_available() and prefer_small:
        return "yolo26s.pt"
    return "yolo26n.pt"


def get_optimal_train_profile(imgsz: int = 640) -> dict[str, Any]:
    """Auto-tune batch size, workers, and AMP based on Colab GPU VRAM (A100 / L4 / T4 / CPU)."""
    if not torch.cuda.is_available():
        return {"device": "cpu", "batch": 8, "workers": 2, "amp": False, "gpu_name": "CPU", "vram_gb": 0.0}

    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / (1024 ** 3)
    scale = (640.0 / float(imgsz)) ** 2

    if vram_gb >= 36.0:  # A100 (40GB / 80GB)
        base_batch, workers = 64, 8
    elif vram_gb >= 20.0:  # L4 (24GB)
        base_batch, workers = 40, 8
    elif vram_gb >= 12.0:  # T4 (15GB)
        base_batch, workers = 32, 4
    else:
        base_batch, workers = 16, 4

    cpu_cnt = os.cpu_count() or 2
    workers = min(workers, max(2, cpu_cnt))
    batch = max(8, int(base_batch * scale))
    return {
        "device": 0,
        "batch": batch,
        "workers": workers,
        "amp": True,
        "gpu_name": props.name,
        "vram_gb": round(vram_gb, 1),
    }


def train_yolo26_stage1(
    data_yaml: Path,
    project_dir: Path = RUNS_DIR,
    epochs: int = 50,
    batch: int | None = None,
    imgsz: int = 640,
) -> Path:
    """
    Stage 1: Train YOLO26 on the full combined dataset with One-to-Many NMS head (nms=True).
    """
    weights = select_yolo26_backbone(prefer_small=True)
    profile = get_optimal_train_profile(imgsz=imgsz)
    model = YOLO(weights)

    train_kwargs: dict[str, Any] = {
        "data": str(data_yaml),
        "epochs": epochs,
        "imgsz": imgsz,
        "batch": batch if batch is not None else profile["batch"],
        "device": profile["device"],
        "amp": profile["amp"],
        "project": str(project_dir),
        "name": "yolo26_stage1",
        "exist_ok": True,
        "optimizer": "AdamW",
        "lr0": 0.005,
        "cos_lr": True,
        # Keep One-to-Many head with NMS for dense Vietnamese intersections
        "nms": True,
        "max_det": 500,
        # Mild HSV augmentation for headlight glare & shadows
        "hsv_h": 0.015,
        "hsv_s": 0.50,
        "hsv_v": 0.40,
        "fliplr": 0.5,
        "flipud": 0.0,
        "mosaic": 0.8,
        "close_mosaic": 10,
        "workers": profile["workers"],
    }

    print(f"[Stage 1] Profile: {profile['gpu_name']} ({profile['vram_gb']} GB) | batch={train_kwargs['batch']} | workers={train_kwargs['workers']} | amp={train_kwargs['amp']}")
    model.train(**train_kwargs)
    best_pt = project_dir / "yolo26_stage1" / "weights" / "best.pt"
    print(f"[Stage 1] Best weights saved at: {best_pt}")
    return best_pt


def train_yolo26_stage2_finetune(
    stage1_weights: Path,
    stage2_yaml: Path,
    project_dir: Path = RUNS_DIR,
    epochs: int = 25,
    batch: int | None = None,
    imgsz: int = 640,
) -> Path:
    """
    Stage 2 Fine-tuning: Adapt Stage 1 weights to nighttime, CCTV angles, and minority classes (bus/truck).
    """
    profile = get_optimal_train_profile(imgsz=imgsz)
    model = YOLO(str(stage1_weights))

    finetune_kwargs: dict[str, Any] = {
        "data": str(stage2_yaml),
        "epochs": epochs,
        "imgsz": imgsz,
        "batch": batch if batch is not None else profile["batch"],
        "device": profile["device"],
        "amp": profile["amp"],
        "project": str(project_dir),
        "name": "yolo26_stage2_finetune",
        "exist_ok": True,
        "optimizer": "AdamW",
        "lr0": 0.001,  # Low LR to preserve Stage 1 features while boosting bus/truck/night
        "lrf": 0.01,
        "cos_lr": True,
        "nms": True,
        "max_det": 500,
        "hsv_h": 0.015,
        "hsv_s": 0.50,
        "hsv_v": 0.45,
        "fliplr": 0.5,
        "flipud": 0.0,
        "mosaic": 0.5,
        "close_mosaic": 5,
        "workers": profile["workers"],
    }

    print(f"[Stage 2] Profile: {profile['gpu_name']} ({profile['vram_gb']} GB) | batch={finetune_kwargs['batch']} | workers={finetune_kwargs['workers']} | amp={finetune_kwargs['amp']}")
    model.train(**finetune_kwargs)
    best_pt = project_dir / "yolo26_stage2_finetune" / "weights" / "best.pt"
    print(f"[Stage 2] Fine-tuned weights saved at: {best_pt}")
    return best_pt


# Example Colab execution:
# stage1_best = train_yolo26_stage1(PROCESSED_DIR / "dataset_stage1.yaml", epochs=50)
# stage2_best = train_yolo26_stage2_finetune(stage1_best, PROCESSED_DIR / "dataset_stage2_finetune.yaml", epochs=25)
print("2-Stage YOLO26 training functions ready (backbone:", select_yolo26_backbone(), "| profile:", get_optimal_train_profile(), ").")


---
## Phần 5: Theo dấu Đa đối tượng (ByteTrack vs BoT-SORT), Trích xuất Điểm chạm đất & Xuất `tracks_stream.jsonl`

### 5.1. Nguyên lý Điểm chạm đất (Ground-Plane Footpoint)
Phép chiếu Homography $H$ ($3 \times 3$) chỉ chính xác trên mặt phẳng đường ($Z = 0$). Nếu dùng tâm hộp giới hạn $(x_c, y_c)$, các xe có chiều cao lớn như `bus` hoặc `truck` sẽ bị sai số cao độ (Parallax Error), khiến vị trí chiếu lên bản đồ 2D BEV bị trôi xa khỏi làn thực tế.  
Do đó, Người A trích xuất **trung điểm cạnh đáy** của hộp giới hạn (vị trí lốp xe chạm mặt nhựa đường):
$$u_{\text{đáy}} = \frac{x_1 + x_2}{2}, \quad v_{\text{đáy}} = y_2$$

### 5.2. Đối chứng Bộ theo dấu ByteTrack vs BoT-SORT
- **ByteTrack (`bytetrack_vn.yaml`):** Ghép cả các hộp tin cậy cao (`track_high_thresh=0.45`) và hộp tin cậy thấp (`track_low_thresh=0.10`) để giữ ID xe máy khi bị che khuất nhẹ.
- **BoT-SORT (`botsort_vn.yaml`):** Bổ sung bù chuyển động camera (`gmc_method: sparseOptFlow`) và giữ bộ đệm quỹ đạo dài hơn (`track_buffer=45`) để giảm nhảy ID (`IDSW`) khi xe máy đi sau xe buýt/xe tải.


In [ ]:
import time


def write_tracker_configs(cfg_dir: Path) -> tuple[Path, Path]:
    """Create tuned ByteTrack and BoT-SORT YAML configs for Vietnamese urban traffic."""
    cfg_dir.mkdir(parents=True, exist_ok=True)

    bytetrack_cfg = {
        "tracker_type": "bytetrack",
        "track_high_thresh": 0.45,
        "track_low_thresh": 0.10,
        "new_track_thresh": 0.55,
        "track_buffer": 30,
        "match_thresh": 0.80,
        "fuse_score": True,
    }
    botsort_cfg = {
        "tracker_type": "botsort",
        "track_high_thresh": 0.45,
        "track_low_thresh": 0.10,
        "new_track_thresh": 0.55,
        "track_buffer": 45,
        "match_thresh": 0.80,
        "fuse_score": True,
        "gmc_method": "sparseOptFlow",
        "proximity_thresh": 0.50,
        "appearance_thresh": 0.25,
        "with_reid": False,
    }

    bt_path = cfg_dir / "bytetrack_vn.yaml"
    bs_path = cfg_dir / "botsort_vn.yaml"
    with open(bt_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(bytetrack_cfg, f, sort_keys=False)
    with open(bs_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(botsort_cfg, f, sort_keys=False)

    return bt_path, bs_path


def compute_footpoint_uv(bbox_xyxy: list[float]) -> list[float]:
    """Compute ground-plane footpoint [u_bottom, v_bottom] = [(x1 + x2) / 2, y2]."""
    x1, _, x2, y2 = bbox_xyxy
    return [float((x1 + x2) / 2.0), float(y2)]


def run_mot_and_export_stream(
    weights_path: Path | str,
    video_path: Path | str,
    tracker_yaml: Path | str,
    output_jsonl: Path,
    camera_id: str = "cam_hcmc_01",
    conf: float = 0.25,
    iou: float = 0.50,
    imgsz: int = 640,
    export_parquet: bool = True,
) -> dict[str, Any]:
    """
    Run YOLO26 + MOT (model.track(..., persist=True)) on a video stream,
    compute ground-plane footpoints, and export tracks_stream.jsonl (+ .parquet).
    """
    import pandas as pd

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise FileNotFoundError(f"Cannot open video: {video_path}")
    fps = float(cap.get(cv2.CAP_PROP_FPS) or 30.0)
    if fps <= 0.0:
        fps = 30.0
    cap.release()

    model = YOLO(str(weights_path))
    output_jsonl.parent.mkdir(parents=True, exist_ok=True)

    records: list[dict[str, Any]] = []
    t0 = time.perf_counter()
    frame_idx = 0

    results_gen = model.track(
        source=str(video_path),
        tracker=str(tracker_yaml),
        persist=True,
        stream=True,
        conf=conf,
        iou=iou,
        imgsz=imgsz,
        nms=True,
        max_det=500,
        verbose=False,
    )

    for res in results_gen:
        timestamp_ms = round((frame_idx / fps) * 1000.0, 3)
        boxes = res.boxes
        if boxes is not None and boxes.id is not None and len(boxes) > 0:
            xyxy_arr = boxes.xyxy.cpu().numpy()
            cls_arr = boxes.cls.cpu().numpy().astype(int)
            conf_arr = boxes.conf.cpu().numpy()
            id_arr = boxes.id.cpu().numpy().astype(int)

            for xyxy, cls_id, score, tid in zip(xyxy_arr, cls_arr, conf_arr, id_arr):
                if cls_id not in CANONICAL_CLASSES or tid < 0:
                    continue
                bbox = [round(float(v), 2) for v in xyxy.tolist()]
                foot_uv = [round(v, 2) for v in compute_footpoint_uv(bbox)]
                records.append(
                    {
                        "camera_id": str(camera_id),
                        "frame_id": int(frame_idx),
                        "timestamp_ms": float(timestamp_ms),
                        "fps": float(round(fps, 3)),
                        "track_id": int(tid),
                        "class_id": int(cls_id),
                        "class_name": CANONICAL_CLASSES[int(cls_id)],
                        "conf": float(round(float(score), 4)),
                        "bbox_xyxy": bbox,
                        "footpoint_uv": foot_uv,
                    }
                )
        frame_idx += 1

    elapsed_s = max(time.perf_counter() - t0, 1e-6)
    proc_fps = frame_idx / elapsed_s
    ms_per_frame = (elapsed_s / max(frame_idx, 1)) * 1000.0

    with open(output_jsonl, "w", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

    parquet_path = None
    if export_parquet and records:
        parquet_path = output_jsonl.with_suffix(".parquet")
        pd.DataFrame(records).to_parquet(parquet_path, index=False)

    summary = {
        "video": str(video_path),
        "tracker": Path(tracker_yaml).name,
        "frames_processed": frame_idx,
        "total_track_records": len(records),
        "unique_track_ids": len({r["track_id"] for r in records}),
        "processing_fps": round(proc_fps, 2),
        "ms_per_frame": round(ms_per_frame, 2),
        "output_jsonl": str(output_jsonl),
        "output_parquet": str(parquet_path) if parquet_path else None,
    }
    print("MOT Export Summary:", summary)
    return summary


def validate_tracks_stream(jsonl_path: Path, schema_path: Path = SCHEMA_PATH) -> int:
    """Validate every record in tracks_stream.jsonl against contracts/schema.json."""
    with open(schema_path, "r", encoding="utf-8") as sf:
        schema = json.load(sf)

    validator = jsonschema.Draft7Validator(schema)
    count = 0
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            validator.validate(rec)
            assert CANONICAL_CLASSES[rec["class_id"]] == rec["class_name"], (
                f"Line {line_no}: class_id/class_name mismatch"
            )
            x1, _, x2, y2 = rec["bbox_xyxy"]
            u_exp, v_exp = (x1 + x2) / 2.0, y2
            u_act, v_act = rec["footpoint_uv"]
            assert abs(u_act - u_exp) < 0.05 and abs(v_act - v_exp) < 0.05, (
                f"Line {line_no}: footpoint_uv mismatch"
            )
            count += 1

    print(f"Schema validation PASSED for {count} records in {jsonl_path.name}.")
    return count


BT_YAML, BS_YAML = write_tracker_configs(WORK_DIR / "trackers")
print("Tracker configs written:", BT_YAML, BS_YAML)


---
## Phần 6: Đánh giá Kiểm thử & Xuất Bảng Số liệu Chương 6 (Evaluation)

### 6.1. Đánh giá Tầng Detection & Phân lát Ngày vs. Đêm (Slice Evaluation)
Thực hiện đánh giá trên 2 tập phân lát độc lập (`dataset_daytime_val.yaml` và `dataset_nighttime_val.yaml`) để kiểm chứng:
- $\text{mAP@50}_{\text{daytime}} \ge 0.80$
- $\text{Recall}_{\text{motorbike, nighttime}} \ge 0.75$
- Xuất bảng chỉ số chi tiết theo từng lớp (`motorbike`, `car`, `bus`, `truck`) gồm Precision, Recall, F1-Score, mAP@50, mAP@50-95 cùng các đồ thị `BoxPR_curve`, `BoxF1_curve` và `confusion_matrix`.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


def evaluate_detection_slice(
    weights_path: Path | str,
    data_yaml: Path | str,
    slice_name: str,
    imgsz: int = 640,
) -> tuple[pd.DataFrame, Path]:
    """Run YOLO26 validation on a dataset slice and return per-class & overall metrics."""
    model = YOLO(str(weights_path))
    metrics = model.val(
        data=str(data_yaml),
        imgsz=imgsz,
        split="val",
        nms=True,
        max_det=500,
        project=str(RUNS_DIR / "eval"),
        name=f"val_{slice_name}",
        exist_ok=True,
        plots=True,
        verbose=False,
    )

    rows = []
    box = metrics.box
    for cid, cname in CANONICAL_CLASSES.items():
        if cid < len(box.ap50):
            p = float(box.p[cid])
            r = float(box.r[cid])
            f1 = (2 * p * r / (p + r)) if (p + r) > 0 else 0.0
            rows.append(
                {
                    "Slice": slice_name,
                    "Class": cname,
                    "Precision": round(p, 4),
                    "Recall": round(r, 4),
                    "F1-Score": round(f1, 4),
                    "mAP@50": round(float(box.ap50[cid]), 4),
                    "mAP@50-95": round(float(box.ap[cid]), 4),
                }
            )

    mp = float(box.mp)
    mr = float(box.mr)
    mf1 = (2 * mp * mr / (mp + mr)) if (mp + mr) > 0 else 0.0
    rows.append(
        {
            "Slice": slice_name,
            "Class": "ALL (Macro)",
            "Precision": round(mp, 4),
            "Recall": round(mr, 4),
            "F1-Score": round(mf1, 4),
            "mAP@50": round(float(box.map50), 4),
            "mAP@50-95": round(float(box.map), 4),
        }
    )

    df = pd.DataFrame(rows)
    save_dir = Path(metrics.save_dir)
    return df, save_dir


def run_day_night_slice_evaluation(
    weights_path: Path | str,
    daytime_yaml: Path | str,
    nighttime_yaml: Path | str,
) -> pd.DataFrame:
    """Compare Detection performance between Daytime and Nighttime slices & verify targets."""
    df_day, day_plots = evaluate_detection_slice(weights_path, daytime_yaml, "Daytime")
    df_night, night_plots = evaluate_detection_slice(weights_path, nighttime_yaml, "Nighttime")

    combined = pd.concat([df_day, df_night], ignore_index=True)
    out_csv = OUTPUTS_DIR / "chapter6_detection_slice_evaluation.csv"
    combined.to_csv(out_csv, index=False, encoding="utf-8-sig")

    day_map50 = float(df_day.loc[df_day["Class"] == "ALL (Macro)", "mAP@50"].iloc[0])
    night_moto_recall = float(df_night.loc[df_night["Class"] == "motorbike", "Recall"].iloc[0])

    print(f"Daytime mAP@50 = {day_map50:.4f} (Target >= 0.80)")
    print(f"Nighttime Motorbike Recall = {night_moto_recall:.4f} (Target >= 0.75)")
    print(f"Saved Detection Slice Table to: {out_csv}")
    print(f"Plots available at: {day_plots} and {night_plots}")
    return combined


def display_yolo_eval_plots(eval_dir: Path) -> None:
    """Display BoxPR_curve, BoxF1_curve, and confusion_matrix generated by Ultralytics val()."""
    plot_names = ["BoxPR_curve.png", "BoxF1_curve.png", "confusion_matrix.png"]
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, pname in zip(axes, plot_names):
        p = eval_dir / pname
        if p.exists():
            img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
            ax.imshow(img)
            ax.set_title(pname)
        else:
            ax.set_title(f"{pname} (not found)")
        ax.axis("off")
    plt.tight_layout()
    plt.show()


print("Detection & Day/Night Slice Evaluation functions ready.")


### 6.2. Đánh giá Tầng MOT trên Tập Zenodo VnTraffic (MOTA, IDF1, HOTA, IDSW, FPS)
Trên tập dữ liệu **VnTraffic** đã có sẵn Ground-Truth MOT (`Frame ID`, `Track ID`, `Bounding Box`), ta đối chứng trực tiếp hai bộ theo dấu **ByteTrack** và **BoT-SORT**:
- **MOTA** (*Multiple Object Tracking Accuracy*): Đánh giá tổng hợp sai sót $\text{FP}, \text{FN}, \text{IDSW}$.
- **IDF1**: Đo độ nhất quán của mã định danh `track_id` theo thời gian (mục tiêu $\ge 0.75$).
- **HOTA** (*Higher Order Tracking Accuracy*): Trung bình hình học giữa độ chính xác phát hiện ($\text{DetA}$) và độ chính xác liên kết quỹ đạo ($\text{AssA}$) qua các ngưỡng $\alpha \in \{0.05, 0.10, \dots, 0.95\}$ (mục tiêu $\ge 0.60$).
- **IDSW** (*ID Switches*) & **FPS**: Số lần nhảy ID và tốc độ thực thi thực tế.


In [ ]:
import motmetrics as mm
from scipy.optimize import linear_sum_assignment


def compute_iou_matrix(boxes_a: np.ndarray, boxes_b: np.ndarray) -> np.ndarray:
    """Compute pairwise IoU matrix between two (N, 4) and (M, 4) xyxy arrays."""
    if len(boxes_a) == 0 or len(boxes_b) == 0:
        return np.zeros((len(boxes_a), len(boxes_b)), dtype=np.float64)

    ax1, ay1, ax2, ay2 = boxes_a[:, 0:1], boxes_a[:, 1:2], boxes_a[:, 2:3], boxes_a[:, 3:4]
    bx1, by1, bx2, by2 = boxes_b[:, 0], boxes_b[:, 1], boxes_b[:, 2], boxes_b[:, 3]

    inter_w = np.maximum(0.0, np.minimum(ax2, bx2) - np.maximum(ax1, bx1))
    inter_h = np.maximum(0.0, np.minimum(ay2, by2) - np.maximum(ay1, by1))
    inter_area = inter_w * inter_h

    area_a = np.maximum(0.0, ax2 - ax1) * np.maximum(0.0, ay2 - ay1)
    area_b = np.maximum(0.0, bx2 - bx1) * np.maximum(0.0, by2 - by1)
    union = area_a + area_b - inter_area + 1e-9
    return inter_area / union


def load_vntraffic_gt_mot(gt_txt_path: Path) -> dict[int, list[tuple[int, list[float]]]]:
    """
    Load Ground-Truth MOT annotations (MOTChallenge format: frame,id,x,y,w,h,...)
    into {frame_id: [(track_id, [x1, y1, x2, y2]), ...]}.
    """
    gt_by_frame: dict[int, list[tuple[int, list[float]]]] = {}
    with open(gt_txt_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = [p.strip() for p in line.strip().split(",")]
            if len(parts) < 6:
                continue
            frame_id = int(float(parts[0])) - 1  # Convert 1-based MOTChallenge to 0-based
            track_id = int(float(parts[1]))
            x, y, w, h = map(float, parts[2:6])
            gt_by_frame.setdefault(frame_id, []).append((track_id, [x, y, x + w, y + h]))
    return gt_by_frame


def load_predictions_from_jsonl(jsonl_path: Path) -> dict[int, list[tuple[int, list[float]]]]:
    """Load predicted tracks from tracks_stream.jsonl into {frame_id: [(track_id, bbox_xyxy), ...]}."""
    pred_by_frame: dict[int, list[tuple[int, list[float]]]] = {}
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            rec = json.loads(line)
            pred_by_frame.setdefault(int(rec["frame_id"]), []).append(
                (int(rec["track_id"]), [float(v) for v in rec["bbox_xyxy"]])
            )
    return pred_by_frame


def compute_hota_score(
    gt_by_frame: dict[int, list[tuple[int, list[float]]]],
    pred_by_frame: dict[int, list[tuple[int, list[float]]]],
) -> float:
    """
    Compute HOTA (Higher Order Tracking Accuracy, Luiten et al. 2021) across IoU thresholds
    alpha in {0.05, 0.10, ..., 0.95}.
    """
    alphas = np.linspace(0.05, 0.95, 19)
    all_frames = sorted(set(gt_by_frame.keys()) | set(pred_by_frame.keys()))
    hota_per_alpha: list[float] = []

    for alpha in alphas:
        tp = 0
        fp = 0
        fn = 0
        # Track co-occurrence counts for AssA: (gt_id, pred_id) -> count
        pair_counts: dict[tuple[int, int], int] = {}
        gt_id_counts: dict[int, int] = {}
        pred_id_counts: dict[int, int] = {}
        matched_pairs: list[tuple[int, int]] = []

        for fid in all_frames:
            gts = gt_by_frame.get(fid, [])
            prs = pred_by_frame.get(fid, [])
            for gid, _ in gts:
                gt_id_counts[gid] = gt_id_counts.get(gid, 0) + 1
            for pid, _ in prs:
                pred_id_counts[pid] = pred_id_counts.get(pid, 0) + 1

            if not gts and not prs:
                continue
            if not gts:
                fp += len(prs)
                continue
            if not prs:
                fn += len(gts)
                continue

            gt_boxes = np.array([b for _, b in gts], dtype=np.float64)
            pr_boxes = np.array([b for _, b in prs], dtype=np.float64)
            iou_mat = compute_iou_matrix(gt_boxes, pr_boxes)

            row_idx, col_idx = linear_sum_assignment(-iou_mat)
            matched_gt = set()
            matched_pr = set()
            for r, c in zip(row_idx, col_idx):
                if iou_mat[r, c] >= alpha:
                    gid = gts[r][0]
                    pid = prs[c][0]
                    matched_gt.add(r)
                    matched_pr.add(c)
                    tp += 1
                    pair_counts[(gid, pid)] = pair_counts.get((gid, pid), 0) + 1
                    matched_pairs.append((gid, pid))

            fn += len(gts) - len(matched_gt)
            fp += len(prs) - len(matched_pr)

        if tp == 0:
            hota_per_alpha.append(0.0)
            continue

        det_a = tp / (tp + fp + fn)
        ass_sum = 0.0
        for gid, pid in matched_pairs:
            tpa = pair_counts[(gid, pid)]
            fna = gt_id_counts[gid] - tpa
            fpa = pred_id_counts[pid] - tpa
            ass_sum += tpa / (tpa + fna + fpa)
        ass_a = ass_sum / tp
        hota_per_alpha.append(float(np.sqrt(det_a * ass_a)))

    return float(np.mean(hota_per_alpha))


def evaluate_mot_benchmark(
    gt_by_frame: dict[int, list[tuple[int, list[float]]]],
    pred_by_frame: dict[int, list[tuple[int, list[float]]]],
    tracker_name: str,
    processing_fps: float,
) -> dict[str, Any]:
    """Compute MOTA, IDF1, HOTA, IDSW, and FPS using motmetrics + HOTA."""
    acc = mm.MOTAccumulator(auto_id=True)
    all_frames = sorted(set(gt_by_frame.keys()) | set(pred_by_frame.keys()))

    for fid in all_frames:
        gts = gt_by_frame.get(fid, [])
        prs = pred_by_frame.get(fid, [])
        gt_ids = [g[0] for g in gts]
        pr_ids = [p[0] for p in prs]

        if gts and prs:
            gt_boxes = np.array([g[1] for g in gts], dtype=np.float64)
            pr_boxes = np.array([p[1] for p in prs], dtype=np.float64)
            iou_mat = compute_iou_matrix(gt_boxes, pr_boxes)
            dist_mat = np.where(iou_mat >= 0.5, 1.0 - iou_mat, np.nan)
        else:
            dist_mat = np.empty((len(gt_ids), len(pr_ids)))

        acc.update(gt_ids, pr_ids, dist_mat)

    mh = mm.metrics.create()
    summary = mh.compute(
        acc,
        metrics=["mota", "idf1", "num_switches", "mostly_tracked", "num_false_positives", "num_misses"],
        name=tracker_name,
    )
    hota = compute_hota_score(gt_by_frame, pred_by_frame)

    return {
        "Tracker": tracker_name,
        "MOTA": round(float(summary["mota"].iloc[0]), 4),
        "IDF1": round(float(summary["idf1"].iloc[0]), 4),
        "HOTA": round(hota, 4),
        "IDSW": int(summary["num_switches"].iloc[0]),
        "FP": int(summary["num_false_positives"].iloc[0]),
        "FN": int(summary["num_misses"].iloc[0]),
        "FPS": round(float(processing_fps), 2),
        "Latency_ms": round(1000.0 / max(processing_fps, 1e-6), 2),
    }


def benchmark_bytetrack_vs_botsort(
    weights_path: Path | str,
    vntraffic_video: Path | str,
    vntraffic_gt_txt: Path | str,
    camera_id: str = "vntraffic_hanoi_01",
) -> pd.DataFrame:
    """Run ByteTrack vs BoT-SORT on VnTraffic and export Chapter 6 MOT comparison table."""
    gt_by_frame = load_vntraffic_gt_mot(Path(vntraffic_gt_txt))
    rows: list[dict[str, Any]] = []

    for tracker_label, tracker_yaml in [("ByteTrack", BT_YAML), ("BoT-SORT", BS_YAML)]:
        out_jsonl = OUTPUTS_DIR / f"tracks_vntraffic_{tracker_label.lower().replace('-', '')}.jsonl"
        run_info = run_mot_and_export_stream(
            weights_path=weights_path,
            video_path=vntraffic_video,
            tracker_yaml=tracker_yaml,
            output_jsonl=out_jsonl,
            camera_id=camera_id,
        )
        validate_tracks_stream(out_jsonl)
        pred_by_frame = load_predictions_from_jsonl(out_jsonl)
        metrics_row = evaluate_mot_benchmark(
            gt_by_frame=gt_by_frame,
            pred_by_frame=pred_by_frame,
            tracker_name=tracker_label,
            processing_fps=run_info["processing_fps"],
        )
        rows.append(metrics_row)

    df_mot = pd.DataFrame(rows)
    out_csv = OUTPUTS_DIR / "chapter6_mot_comparison.csv"
    df_mot.to_csv(out_csv, index=False, encoding="utf-8-sig")
    print("Chapter 6 MOT Benchmark Table:")
    print(df_mot.to_string(index=False))
    return df_mot


print("VnTraffic MOT Evaluation (MOTA, IDF1, HOTA, IDSW, FPS) pipeline ready.")


---
## Phần 7: Chạy Toàn Bộ Pipeline (End-to-End Execution trên Google Colab)

Cell dưới đây sẽ tự động:
1. Tải bộ dữ liệu chủ lực Kaggle bằng `kagglehub` (không cần `kaggle.json`), tải Roboflow (nếu có `ROBOFLOW_API_KEY`, không có sẽ tự bỏ qua) và tải Zenodo `VnTraffic`.
2. Chuẩn hóa toàn bộ nhãn về 4 lớp cố định, lọc hộp `< 8px`, lọc trùng SSIM và chia tập theo cụm cảnh quay.
3. Huấn luyện **YOLO26 2 giai đoạn** tự động tối ưu `batch` & `workers` theo GPU Colab (A100 / L4 / T4).
4. Đánh giá phân lát Ngày vs. Đêm, đối chứng **ByteTrack vs. BoT-SORT** trên `VnTraffic` và xuất `tracks_stream.jsonl`.

In [ ]:
RUN_FULL_PIPELINE = True
STAGE1_EPOCHS = 50
STAGE2_EPOCHS = 25
SSIM_FRAME_STRIDE = 1


def _collect_kaggle_records(kaggle_root: Path) -> list[dict[str, Any]]:
    """Scan Kaggle vietnamese-vehicles-dataset (daytime/nighttime) and remap labels."""
    records: list[dict[str, Any]] = []
    img_exts = {".jpg", ".jpeg", ".png", ".bmp"}
    default_names = CANONICAL_CLASSES

    all_imgs = [p for p in kaggle_root.rglob("*") if p.suffix.lower() in img_exts]
    kept_imgs = filter_near_duplicates_ssim(all_imgs, ssim_threshold=0.97, frame_stride=SSIM_FRAME_STRIDE)
    for img_path in kept_imgs:
        lbl_path = img_path.with_suffix(".txt")
        if not lbl_path.exists():
            # Check parallel labels/ folder if images/labels layout is used
            alt_lbl = Path(str(img_path).replace(f"{os.sep}images{os.sep}", f"{os.sep}labels{os.sep}")).with_suffix(".txt")
            if alt_lbl.exists():
                lbl_path = alt_lbl
        cleaned, _ = remap_and_clean_labels(lbl_path, img_path, default_names, min_box_px=8.0)
        if not cleaned:
            continue
        lower_parts = "/".join(p.lower() for p in img_path.parts)
        lighting = "nighttime" if "night" in lower_parts else "daytime"
        records.append({"image": img_path, "labels": cleaned, "lighting": lighting, "source": "kaggle"})
    return records


def _collect_roboflow_records(rf_dirs: dict[str, Path]) -> list[dict[str, Any]]:
    """Scan downloaded Roboflow YOLO datasets and remap labels."""
    records: list[dict[str, Any]] = []
    img_exts = {".jpg", ".jpeg", ".png", ".bmp"}
    for name, rf_root in rf_dirs.items():
        data_yaml = rf_root / "data.yaml"
        src_names: list[str] | dict[int, str] = CANONICAL_CLASSES
        if data_yaml.exists():
            with open(data_yaml, "r", encoding="utf-8") as f:
                cfg = yaml.safe_load(f) or {}
                if "names" in cfg:
                    src_names = cfg["names"]
        for img_path in sorted(rf_root.rglob("*")):
            if img_path.suffix.lower() not in img_exts:
                continue
            lbl_path = Path(str(img_path).replace(f"{os.sep}images{os.sep}", f"{os.sep}labels{os.sep}")).with_suffix(".txt")
            if not lbl_path.exists():
                lbl_path = img_path.with_suffix(".txt")
            cleaned, _ = remap_and_clean_labels(lbl_path, img_path, src_names, min_box_px=8.0)
            if not cleaned:
                continue
            records.append({"image": img_path, "labels": cleaned, "lighting": "cctv", "source": name})
    return records


if RUN_FULL_PIPELINE:
    print("=== [1/5] Downloading datasets ===")
    kaggle_root = download_kaggle_primary_dataset(RAW_DIR)
    rf_dirs = download_roboflow_datasets(RAW_DIR, include_backup=False)
    vntraffic_dir = download_zenodo_vntraffic(RAW_DIR)

    print("=== [2/5] Preprocessing & Anti-Data-Leakage Splitting ===")
    all_records = _collect_kaggle_records(kaggle_root) + _collect_roboflow_records(rf_dirs)
    print(f"Total cleaned image records: {len(all_records):,}")
    yaml_paths = split_dataset_by_scene_clusters(all_records, PROCESSED_DIR)

    print("=== [3/5] Training YOLO26 (2-Stage, One-to-Many nms=True) ===")
    stage1_best = train_yolo26_stage1(yaml_paths["stage1"], epochs=STAGE1_EPOCHS)
    stage2_best = train_yolo26_stage2_finetune(stage1_best, yaml_paths["stage2"], epochs=STAGE2_EPOCHS)

    print("=== [4/5] Detection & Day/Night Slice Evaluation ===")
    slice_df = run_day_night_slice_evaluation(stage2_best, yaml_paths["daytime_val"], yaml_paths["nighttime_val"])

    print("=== [5/5] VnTraffic MOT Benchmark (ByteTrack vs BoT-SORT) & Export ===")
    vn_video = next(vntraffic_dir.rglob("VNTraffic_Original-video.mp4"))
    vn_gt_txt = next(vntraffic_dir.rglob("VNTraffic_GroundTruth.txt"))
    mot_df = benchmark_bytetrack_vs_botsort(stage2_best, vn_video, vn_gt_txt)

    # Copy best tracker stream to canonical tracks_stream.jsonl
    best_tracker = mot_df.sort_values("IDF1", ascending=False).iloc[0]["Tracker"].lower().replace("-", "")
    best_jsonl = OUTPUTS_DIR / f"tracks_vntraffic_{best_tracker}.jsonl"
    final_jsonl = OUTPUTS_DIR / "tracks_stream.jsonl"
    shutil.copyfile(best_jsonl, final_jsonl)
    print(f"=== DONE! Best tracker: {best_tracker} -> {final_jsonl} ===")
